# Debugging Player Features vs Elephant Gambit
Extracts target player features and the Elephant Gambit features to show them both side by side as output.

In [6]:
import chess.engine
import pandas as pd
from utils.chesscom_repository import ChessComRepository
from utils.game_enrichment_transformer import GameEnrichmentTransformer
from utils.opening_repository import OpeningRepository
from utils.player_statistics import PlayerStatisticsBuilder

USERNAME = "bassisw"
MAX_GAMES = 20
STOCKFISH_PATH = "/opt/homebrew/bin/stockfish"  # Adjust for your architecture if needed
OPENINGS_PATH = "openings_dataset/all.tsv"
OPENING_FEATURE_VECTORS_PATH = "openings_dataset/opening_feature_vectors_sparsity_fix.csv"
GLOBAL_STATISTICS_HPARAMS_PATH = "config/global_statistics_hparams.yaml"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [2]:
print(f"Loading {MAX_GAMES} games for {USERNAME}...")
chesscom_repo = ChessComRepository(USERNAME)
raw_games = []

for game in chesscom_repo.iter_all_games(since_year=2026, since_month=1):
    if "pgn" in game:
        raw_games.append(game)
        if len(raw_games) >= MAX_GAMES:
            break

print(f"Selected {len(raw_games)} games.")

Loading 20 games for bassisw...
Selected 20 games.


In [4]:
print("Enriching games and extracting player features...")
opening_repo = OpeningRepository(OPENINGS_PATH)
enrichment_transformer = GameEnrichmentTransformer(
    stockfish_path=STOCKFISH_PATH,
    opening_repository=opening_repo,
    engine_limit=chess.engine.Limit(depth=10)
)
enriched_games = enrichment_transformer.transform_games(raw_games)

statistics_bundle = PlayerStatisticsBuilder(GLOBAL_STATISTICS_HPARAMS_PATH).build(
    enriched_games, player_name=USERNAME
)

player_vector = statistics_bundle.matcher_ready_player_vector
player_vector_df = pd.DataFrame(
    [{"Feature": k, f"Player ({USERNAME})": v} for k, v in player_vector.items()]
)
display(player_vector_df)

Enriching games and extracting player features...


,Feature,Player (bassisw)
0,tactical_density,0.468571
1,quiet_position_density,0.502857
2,king_safety_risk,0.193951
3,early_castling_tendency,0.375000
4,opposite_side_castling_tendency,0.200000
5,middlegame_complexity,0.519753
6,pawn_structure_sharpness,0.317581
7,material_imbalance,0.606742
8,endgame_likelihood_proxy,0.650000
9,structure_diversity,0.963845


In [7]:
print("Loading opening dataset vectors...")
feature_df = pd.read_csv(OPENING_FEATURE_VECTORS_PATH)

print("Extracting Elephant Gambit feature vector...")
elephant_gambit = feature_df[
    feature_df["opening_name"].str.contains("Elephant Gambit", na=False, case=False)
].copy()

# Ensure we just keep vector columns matching the same keys, if you want side by side comparison
display(elephant_gambit.T)

Loading opening dataset vectors...
Extracting Elephant Gambit feature vector...


,40
opening_name,Elephant Gambit
line_count,4
eco_values,C40
representative_pgn,1. e4 e5 2. Nf3 d5 3. Nxe5 dxe4 4. Bc4 Qg5
representative_uci,e2e4 e7e5 g1f3 d7d5 f3e5 d5e4 f1c4 d8g5
tactical_density,0.144241
quiet_position_density,0.875245
king_safety_risk,0.024
early_castling_tendency,0.50375
opposite_side_castling_tendency,0.103125
